# VSLAM – Visual Simultaneous Localization and Mapping

The main idea of SLAM is to solve the localization and mapping problems jointly.

Task statement: given camera images, estimate the camera trajectory and construct a map of an initially unknown environment.

**VSLAM** – it is essentially SLAM where the primary sensor used for localization and environment mapping is a camera.

**General SLAM** - could also involve the information from LiDAR, IMU, wheel odometry, etc.

# 0. Conventions

$R_{1,2}$ - rotation matrix that transforms points from camera 1 to camera 2 coordinate system.

$R_{2W}$ – rotation matrix that transforms points from camera 2 to world 2 coordinate system.

# 1. Input

- Video sequence
- Camera intrinsic matrix:
    $$K=\begin{bmatrix}f_x&0&c_x\\0&f_y&c_y\\0&0&1\end{bmatrix}$$
- Lens distortion coefficients:
    $$\mathbf D=(k_1,k_2,p_1,p_2,k_3)$$

# 2. Map initialization

1. Extract 2 consecutive frames
2. Undistort images (`cv2.undistort()`)
3. Apply ORB to each frame to extract key-points and their descriptors
4. Match keypoints (brute force matcher + cross-check/Lowe's ratio test)
5. Verify geometric consistency (RANSAC + Essential Matrix / Epipolar geometry):
    1. Randomly select five feature correspondences.
    2. Estimate a candidate essential matrix $E$ using the five-point algorithm.
    3. Evaluate this matrix against all available correspondences.
    4. Classify correspondences as inliers or outliers based on their geometric error.
    5. Repeat with different random subsets.
    6. Select the model with the strongest consensus and estimate/refine the result using its inliers.
6. Recover relative camera pose $(R,t)$ (`cv2.recoverPose()`)
7. Define the world coordinate system $W$ as the origin of the first frame.
8. Perform triangulation to recover 3D points:
    Assumptions:
    1. Triangulation is only reliable when the two viewing rays have sufficient angular separation.
    2. Translation is still expressed in an arbitrary monocular scale. Consequently, the reconstructed 3D map will also have an arbitrary scale.

    Projection matrices:
    $$T_1=K[I|0]$$
    $$T_2=K[R_{12}|t_{12}]$$

    Pinhole camera projection model:
    $$\lambda x = TX$$

    Where:
    - $x = [u, v, 1]^T$ – an image point in homogeneous coordinates.
    - $X = [X,Y,Z,1]^T$ – a 3D point in homogeneous coordinates.
    - $T$ – the camera projection matrix.
    - $\lambda$ – an unknown scale factor.

    For Camera 1:  $\lambda_1 x_1 = T_1X$

    For Camera 2:  $\lambda_2 x_2 = T_2X$

    Direct linear transform (DLT):

    Cross-product of collinear vectors is 0, therefore:

    $x_1 \times (T_1X) = 0$
    $x_2 \times (T_2X) = 0$

    This produces a homogeneous linear system:
    $$A X = 0$$

    We solve it using SVD:
    $$A=U\Sigma V^T$$

    The solution is the right singular vector corresponding to the smallest singular value.

    The resulting point is homogeneous:
    $$\mathbf{X}= \begin{bmatrix} X_h\\Y_h\\Z_h\\W_h \end{bmatrix}$$

    To obtain ordinary 3D coordinates, divide by the fourth component:
    $$
    \boxed{
        \mathbf{P}=\begin{bmatrix}
        X_h/W_h\\
        Y_h/W_h\\
        Z_h/W_h
        \end{bmatrix}
    }
    $$

    DLT could be done via `cv2.triangulatePoints()`
9. Validate reconstructed 3D points:

    1. The depth is positive:
        $$Z_1 > 0$$
        $$Z_2 = (R_{12}P_1 + t_{12})_z > 0$$
    2. Reprojection error – project that reconstructed point back into both images and compare the predicted pixel coordinates with the original observations:

        For camera 1:

        $$\hat{x_1}=\pi(KP_1)$$

        For camera 2:
        $$\hat{x_2}=\pi(K(R_{12}P_1+t_{12}))$$

        Reprojection errors:

        $$e_1=\|x_1-\hat{x}_1\|$$

        $$e_2=\|x_2-\hat{x}_2\|$$

        A large error indicates an unreliable reconstruction.

   3. Paralax – we need sufficient angular separation between the two viewing rays:

        Let $r_1$ and $r_2$ be normalized viewing directions expressed in the same coordinate system.
        Then:

        $$\theta = \cos^{-1}(r_1^Tr_2)$$

        If $\theta$ is very small, depth estimation becomes unstable.
        For the initial experiment, use a threshold such as 1–2 degrees as an experimental starting point, not a universal SLAM requirement.

    4. Reasonable depth – points with extremely large depths relative to the camera baseline may have very uncertain positions.
    We should not use an arbitrary fixed depth threshold for every environment. Instead, consider depth relative to the baseline and the expected scene geometry.

    5. Consistency of the observed feature scales with the estimated 3D distance. ORB features are detected at different pyramid levels. If a landmark is observed at different distances, its apparent image scale should change approximately consistently with the distance ratio. If the observations contradict this relationship, the triangulated point may be unreliable.

   The ORB-SLAM initialization procedure explicitly checks positive depth, parallax and reprojection error before accepting map points.


# 3. Identify the camera location of a new frame based on the existing map

Now, we will estimate camera pose using correspondences between a new image and existing 3D landmarks in the map instead of estimation of correspondences between two images, as we did on the initialization step.

1. We have camera 1 pose: $T_{C_1W}$ and camera 2 pose: $T_{C_2W}$
2. We need to estimate camera 3 pose: $T_{C_3W}$
3. Initial pose estimation – to predict where existing map points should appear in the new image. That significantly reduces the search space for feature matching. A common approach is the constant-velocity motion model:
    $$T_{C_2C_1}=T_{C_1W}^{-1}T_{C_2W}$$
    $$T_{C_3W}^{pred}=T_{C_2W}T_{C_2C_1}$$

    This repeats the previous relative transform. It is an initial guess; image measurements must correct it.
    Unequal timestamps or abrupt motion require a more sophisticated motion model.

4. Find correspondences between 3D landmarks and 2D features:
    1. Take a 3D map point.
    2. Transform it into the predicted camera coordinate system.
    3. Project it onto the image using the camera matrix.
    4. Search for compatible ORB keypoints in a neighborhood around the predicted location.
    5. Compare descriptors using Hamming distance.
    6. Accept good enough matches.
5. Estimate the camera pose using Perspective-n-Point (PnP):

    Now we have a collection of correspondences::
    $$P_i^W \leftrightarrow x_i$$

    Where:
    - $P_i^W$ – a known 3D map point.
    - $x_1$ – its observed 2D image projection.

    PnP solves the reprojection-error minimization problem:
    $$
    \min_{R_{WC},\mathbf{t}_{WC}}
    \sum_i
    \left\|
    \mathbf{x}_i -
    \pi\left(
    K(R_{WC}\mathbf{P}_i^W+\mathbf{t}_{WC})
    \right)
    \right\|^2
    $$

    As a result, we have refined camera 3 pose estimate ($T_{C_3W}$).

    To handle incorrect correspondences we can use PnP with RANSAC:
    1. Select a subset of 3D–2D correspondences.
    2. Estimate a candidate camera pose.
    3. Project the 3D points into the image.
    4. Calculate reprojection errors.
    5. Identify inliers and outliers.
    6. Select the pose with the strongest consensus.

    This task is solved by `cv2.solvePnPRansac()` function.
    After RANSAC, we can refine the pose using all accepted inliers.

    **WARNING!!!** OpenCV returns world-to-camera transformation, but this conspect mostly uses camera-to-world matrices.

    Why to use PnP instead of essential matrix estimation?
    - PnP estimates translation in the coordinate system and scale of the existing map. In VO a translation vector is always of unit length.
    PnP can subsequently estimate movements of 0.5, 2, or 3 arbitrary units relative to that map.
    - In this case we would ignore the existing 3D map → information from previous frames.
    - Relative motion is estimated from the scratch each time → accumulated drift becomes a significant concern.

6. Improve further the pose estimate:
    1. The system retrieves a local map: a subset of keyframes and landmarks relevant to the current camera position.
    2. Search for additional correspondences with those landmarks and performs pose optimization.

    ORB-SLAM performs this as a separate local-map tracking operation after its initial pose estimation.

# 4. Keyframe insertion

1. Keyframe selection. If we add every frame to the map, the data redundancy and computational cost problems arise.

    ORB-SLAM2 uses several conditions for monocular SLAM:
    1. More than 20 frames must have passed since the last global relocalization.
    2. Local mapping must be idle, or more than 20 frames must have passed since the last keyframe insertion.
    3. The current frame must track at least 50 map points.
    4. The current frame must track less than 90% of the map points tracked in the reference keyframe.

    Note 1: These are implementation-specific thresholds, not universal SLAM rules.

    Note 2: camera displacement (e.g., 20 cm) is not used as a keyframe insertion criterion, for several reasons:
    - The camera might rotate substantially without translating.
    - A fixed displacement threshold doesn't account for scene texture.
    - The same movement can produce different visual changes depending on the environment.
    - Monocular SLAM doesn't know the absolute translation scale.

2. The system receives the newly inserted keyframe and updates its relationships with existing map points.

   For each detected ORB feature, there are two possibilities:
    - It corresponds to an existing 3D map point.
    - It doesn't have an associated map point yet.

    For existing map points, the system:
    - Adds another observation.
    - Tracks which camera views contributed to the estimation of each 3D point.
    - Updates the map point's representative descriptor and viewing information.
    - Updates the covisibility graph based on the shared observations.

    **Covisibility graph** – the graph, where two keyframes are connected if they observe common 3D map points.

    Note: ORB-SLAM also maintains a spanning tree connecting keyframes, which is useful for maintaining the map's structural relationships.

    For not existing map points, the system will add them during following steps.

3. Filter recently created map points. Not every triangulated point is reliable.

    Some points may have been created because of:
    - Incorrect descriptor matching.
    - Poor triangulation geometry.
    - Moving objects.
    - Unstable visual features.
    - Insufficient parallax.

    Therefore, we need to evaluate new map points over subsequent frames.

    1. Found ratio. ORB-SLAM2 tracks two quantities:
        - Visible: the number of times a map point was predicted to be visible.
        - Found: number of times it was successfully matched.

        The found ratio is:
            $$r_{\text{found}} =\frac{N_{\text{found}}}{N_{\text{visible}}}$$

        If the ratio is low, the point is suspicious.

    2. Observation count. A point observed by several independent keyframes is generally more trustworthy than one observed only once or twice.

    In ORB-SLAM2, for monocular SLAM, the implementation uses these specific rules:
    - Remove a point if its found ratio is below 0.25.
    - Remove it if, after at least two keyframes have passed since its creation, it has no more than two observations.
    - Stop treating it as a recent point after three keyframes.

    These are ORB-SLAM2 implementation details, not universal thresholds.

   The intuition is straightforward: new landmarks must prove that they are consistently observable before becoming permanent map elements.

4. Create new map points:
    1. Select keyframe's neighbors in the covisibility graph. ORB-SLAM2 selects the best covisible neighbors (20 in monocular mode) for this process.

        Note: neighboring key frames are selected because if two keyframes observe common areas, there is a good chance that we can identify corresponding features between them.

    2. Feature matching with epipolar constraints.

        For each neighboring keyframe, we search for feature correspondences. We already know the relative poses of the two keyframes, so we can calculate their fundamental matrix:
        $$F=K^{-T}EK^{-1}$$
        $$K^{-T}=(K^{-1})^T=(K^{T})^{-1}$$

        Then, for a feature $x_1$ in the first image, its corresponding feature in the second image must lie approximately on the epipolar line:
        $$l_2=Fx_1$$

        A candidate feature that is too far from this line can be rejected.

    3. Triangulate new landmarks.

        For each matched pair $x_1 \leftrightarrow x_2$ calculate $P^W$.
        The resulting 3D point is added to the map and associated with both keyframes.

        Note: Notice that we are now triangulating using two keyframes that may be separated by several frames, rather than necessarily using consecutive video frames. This often gives better parallax.

    4. Validate the new points (the same procedure as in `2. Map initialization` step)

5. Local Bundle Adjustment (BA). Purpose – optimize jointly errors accumulated during tracking and triangulation.

    1. Motivation for joint optimization.

        3D landmark location depends on reconstructed camera positions. If the landmark position is incorrect, its reprojection errors can make camera pose estimates worse for further key frames.

    2. Why local BA instead of global BA?

        Optimizing the entire map after every new keyframe would be computationally expensive.

        Instead, ORB-SLAM2 optimizes a local subset:
        - The current keyframe.
        - Its covisible neighboring keyframes.
        - All map points observed by those keyframes.
        - Other keyframes observing those same points, which can be included as fixed poses.

        The system therefore optimizes a limited portion of the map while keeping the rest unchanged. This makes continuous optimization practical.

        Note: the optimization needs a fixed reference pose (or another gauge constraint) to avoid the entire reconstruction freely changing its coordinate system.

    3. Optimization objective:

        $$\boxed{
        \min_{\{T_i\},\{P_j\}}
        \sum_{i,j}
        \rho\left(
        \left\|
        \mathbf{x}_{ij}
        -
        \pi(T_i,P_j,K)
        \right\|^2
        \right)
        }
        $$

        Here:
        - $T_i$: camera pose of keyframe \(i\).
        - $P_j$: 3D position of landmark \(j\).
        - $\mathbf{x}_{ij}$: observed image position of landmark \(j\) in keyframe \(i\).
        - $\pi$: camera projection function.
        - $\rho$: robust loss function.

        The optimizer changes both camera poses and landmark positions to minimize the total reprojection error.

6. Remove redundant key frames.

    After local BA, the system checks whether some keyframes are redundant.

    ORB-SLAM2 considers a keyframe redundant when approximately 90% of its map points are also observed in at least three other keyframes at the same or finer scale.
   Importantly, this does not mean deleting the corresponding 3D map points. Other keyframes may still observe and preserve those landmarks.

# 5. Loop closure

Motivation:

Suppose the camera:
1. Starts at location A.
2. Walks through a corridor.
3. Turns into another room.
4. Returns to location A.

In reality, the final camera position should coincide with the starting position.
But our estimated trajectory might show that the camera ended up several metres away.
This is accumulated drift.

Intuition: the final camera view may look very similar to the initial camera view.

Target: the current camera pose should be consistent with the previously recorded pose at that location

Steps:
1. Loop detection:
    Feature matching with every previous image is not feasible:
    - computationally expensive.
    - similar-looking places could produce false matches.

    Bag of Visual Words (BoW) is used for place recognition:
    1. Take a large collection of ORB descriptors from many images.
    2. Cluster these descriptors into groups and construct visual vocabulary.

        ORB-SLAM uses DBoW2, which implements hierarchical vocabulary based on a vocabulary tree.
        Each leaf node represents a visual word.

        **Note**: The vocabulary is typically trained offline and then loaded when the SLAM system starts. The system does not need to construct a new vocabulary for every video.
    3. An image can then be represented by a histogram indicating how frequently each visual word occurs. So, each image experiences the transformation:

        $$Image \rightarrow ORB \ descriptors \rightarrow Visual \ words \rightarrow BoW \ vector$$
    4. Calculate similarity score between BoW vectors to detect loop candidates.

        Note: similarity does not prove that the camera has returned to the same physical location
2. Check candidate consistency. The idea is:
    1. Find candidate keyframes using BoW.
    2. Group candidates using their covisibility relationships.
    3. Check whether the same candidate group continues to appear in subsequent keyframes.
    4. Accept a loop hypothesis only when there is sufficient consistency.

    In the original ORB-SLAM implementation, three consecutive consistent loop candidates are required.

    This is useful because a genuine revisited location is likely to produce a sequence of consistent recognitions, rather than one isolated similarity match.

3. Find correspondence between points of the current frame and loop candidate frame:

    $$
    \mathbf{P}_i^{current}
    \longleftrightarrow
    \mathbf{P}_i^{loop}
    $$

4. Geometric verification: estimating the Sim(3) transformation.

    Now we need to determine whether the current map and the old map can actually be aligned geometrically.

    Small errors in relative motion estimation and triangulation can cause scale drift over a long trajectory.
    What is why we cannot just use a rigid transformation and have to use Similarity transformation (in 3d space denoted as Sim(3)).
    It handles differences in translation, rotation, and scale.

    It has seven degrees of freedom:
    - 3 translation parameters.
    - 3 rotation parameters.
    - 1 scale parameter.

    Its mathematical representation is:
    $$
    \boxed{
    \mathbf{P}'=sR\mathbf{P}+\mathbf{t}
    }
    $$

    where:
    - $s$ is a scale factor.
    - $R$ is a rotation matrix.
    - $\mathbf{t}$ is a translation vector.

    Note: This is not the same as recovering absolute metric scale. Monocular SLAM still does not know the real-world distance. It is correcting the relative scale inconsistency between two parts of its reconstruction.

    ORB-SLAM uses a RANSAC procedure with the method of Horn to estimate this transformation.

    The process is conceptually similar to our earlier essential matrix estimation:
    1. Select a subset of 3D–3D correspondences.
    2. Estimate a candidate Sim(3).
    3. Transform the current 3D points.
    4. Calculate alignment errors.
    5. Count the inliers.
    6. Repeat until a sufficiently strong solution is found.

    The resulting transformation also serves as geometric verification of the loop hypothesis.

    After finding a sufficiently good solution, ORB-SLAM optimizes it and performs additional guided matching to find more correspondences.

    Only when the geometric evidence is sufficiently strong, the is loop accepted.

5. Map points fusion: merging duplicated map points:
    1. Apply the estimated Sim(3) correction to the current keyframe.
    2. Propagate the correction to its neighboring keyframes.
    3. Project map points from the old loop region into the corrected current keyframes.
    4. Search for matching features around their projected positions.
    5. Identify duplicated map points.
    6. Fuse the duplicates into a single map point.
    7. Update the covisibility graph.

6. Essential Graph Optimization - we need to distribute the accumulated error across the trajectory.

    1. Extract Essential Graph from covisibility graph.

        Covisibility graph is quite dense. Optimizing every edge would be unnecessarily expensive.
        ORB-SLAM therefore constructs a sparser graph called the Essential Graph.

        It contains:
        1. The spanning tree connecting keyframes.
        2. Strong covisibility connections.
        3. Previously detected loop closure connections.

        All keyframes remain represented, but fewer edges are used.

   2. Graph optimization.

        Each node represents a keyframe pose (camera orientation and location).
        Each edge represents a relative transformation between two keyframes (camera rotation and translation).

        We want to adjust all keyframe poses so that they agree with the relative transformations represented by the edges.
        For now, this requirement is not satisfied for now, because we have transformed the final frame pose.

        Conceptually, the optimization is:
        $$
        \min_{\{T_i\}}
        \sum_{(i,j)\in E}
        \rho\left(
        \left\|
        \mathbf{e}_{ij}(T_i,T_j,Z_{ij})
        \right\|^2
        \right)
        $$

        where:
        - $T_i,T_j$: keyframe poses.
        - $Z_{ij}$: measured relative transformation.
        - $\mathbf{e}_{ij}$: difference between predicted and measured relative transformations.
        - $\rho$: robust loss function.

        For monocular ORB-SLAM, the optimization uses similarity transformations to correct scale drift as well.

        The optimization adjusts the keyframe poses so that the entire trajectory becomes globally consistent.

        ORB-SLAM can additionally perform full Bundle Adjustment after loop closure to further refine the reconstruction.

        After optimizing the Essential Graph, we must also update the 3D map points accordingly.

Conditions to run loop closure:
- Loop closure detection is attempted whenever a new keyframe is processed.
- At least 10 keyframe IDs must elapse before another loop-detection search is allowed.

One architectural detail:

Loop closure runs in a separate thread from tracking and local mapping. Consequently, the camera can continue being tracked while loop candidates are being evaluated. During an actual loop correction, local mapping is temporarily stopped so the map can be updated consistently.

# Global relocalization

**Global relocalization** – the process of recovering the camera's position and orientation within an existing map when normal camera tracking has been lost.

The original ORB-SLAM paper describes the following process:
1. Detect tracking failure. The system cannot estimate a reliable camera pose using the previous frame or local map.
2. Find candidate keyframes. Convert the current frame's ORB descriptors into a Bag-of-Words representation and search the keyframe database for visually similar places.
3. Find 2D–3D correspondences. Match current-frame ORB features to features in candidate keyframes that are associated with existing 3D map points.
4. Estimate camera pose using PnP + RANSAC. This gives a candidate camera pose in the existing map coordinate system.
5. Refine and verify. Optimize the pose, project additional map points into the image, search for more correspondences, and accept relocalization only if enough inliers support the result.
6. Resume normal tracking. Use the recovered pose to track the local map again.
Mathematically, step 4 solves the same problem we've already discussed for normal map-based tracking:
    $$
    \mathbf{x}_i \sim K(R_{CW}\mathbf{P}_i^W+\mathbf{t}_{CW})
    $$

## Tracking failure criteria

- Median reprojection error:
    Reprojection error for a 3D map point $P_i^W$ matched to an observed image point $\mathbf x_i$, is defined:
    $$
    e_i=\left\|\mathbf x_i-\pi\left(K(R_{CW}\mathbf P_i^W+\mathbf t_{CW})\right)\right\|
    $$

- Inlier count:

    But small reprojection error alone is not enough.
    An estimate based on only three or four points may be poorly constrained even if those points fit perfectly.

    ORB-SLAM2 considers tracking successful with $>30$ inliers during regular tracking and with $>50$ inliers shortly after relocalization.

- Inliers ratio:
    $$
    r_{\text{inlier}}=\frac{N_{\text{inliers}}}{N_{\text{matches}}}
    $$
- Other:
    Even 50 geometrically consistent correspondences can produce an unreliable pose if the geometry is poorly conditioned.

    Examples include:
    - Poor spatial distribution: All matched features lie in a small image region.
    - Degenerate geometry: The observed scene provides insufficient constraints for certain camera motions.
    - Repeated patterns: Incorrect correspondences can support a plausible but wrong pose.
    - Incorrect map points: The existing map may contain significant errors.

## Difference to loop closure

Both mechanisms can use Bag-of-Words place recognition, but their objectives differ.
- Global relocalization: Recover the pose of the current camera frame after tracking failure. Primarily uses 2D–3D correspondences and PnP.
- Loop closure: Detect that two map regions represent the same physical place and correct accumulated map/trajectory drift. In monocular ORB-SLAM, this involves Sim(3) estimation and pose-graph optimization.
For example, if tracking is lost at frame 100 and the camera reappears near keyframe $K_{20}$, relocalization can recover the pose of frame 100 using the map points associated with $K_{20}$, without necessarily performing loop closure.

**Note**: ORB-SLAM2's keyframe insertion logic includes a restriction shortly after relocalization. This gives tracking time to stabilize before inserting new keyframes.



# Edge cases

If the camera has no rotation and forward or sideway translation, the epipolar geometry becomes a bit different (see ./computer_vision/2_camera_geometry/5_Fundamental_matrix.ipynb for details).

For monocular SLAM implementation, you do not need a special algorithm simply because two image planes are parallel.

You can still estimate $E$ or $F$, recover the relative camera pose, and triangulate points if sufficient parallax exists.

The important exception is pure rotation or zero translation, where the baseline vanishes and the usual two-view translation estimation becomes degenerate.

For stereo cameras, we often intentionally transform images into the parallel, horizontally aligned configuration. This process is called stereo rectification. It makes matching much easier.

# Implementation details

ORB-SLAM2 has separate threads:
- Tracking: Processes incoming video frames, estimates camera poses, and decides whether to create new keyframes.
- Local Mapping: Processes newly inserted keyframes, creates 3D map points, performs local bundle adjustment, and removes redundant map points and keyframes.
- Loop Closing: Detects and corrects loops.

## BoW data structure

Each stored keyframe has a Bag-of-Words (BoW) representation derived from its ORB descriptors.

The current frame also gets a BoW representation, allowing the system to search for visually similar places.

ORB-SLAM2 uses an inverted index.

Instead of iterating over all keyframes and comparing their descriptors, it stores a mapping:
$$
\text{Visual word}\rightarrow\text{Keyframes containing that word}
$$

Simplified inverted index:

Word A:	$K_1,K_{20},K_{75}$

Word B:	$K_3,K_{20},K_{90}$

Word C:	$K_{20},K_{75}$


Suppose the current frame contains visual words A, B, and C. The $$K_{20}$$ is a strong candidate.

The database retrieves keyframes sharing visual words, filters those with too few common words, computes BoW similarity scores, and uses covisibility neighbors to strengthen candidate scores.

Only the resulting shortlist undergoes expensive feature matching and PnP verification.

### Why not use the trajectory?

We can distinguish two situations:
- Normal tracking: The previous camera pose is reliable, so a motion model and nearby map points are useful.
- Tracking lost: The previous pose is no longer a reliable starting point. A global appearance-based search is more robust to unexpected movement.

You could design a system that first searches keyframes near the last known position and then expands globally if unsuccessful. This may improve efficiency when tracking is lost only briefly, but restricting the search too aggressively risks missing the correct location.

## Local Mapping thread availability

The Local Mapping thread has an internal flag to mark it has reached a state where it is willing to accept another keyframe.

If the current frame would be useful as a keyframe but Local Mapping is busy:
1. Request interruption of local bundle adjustment if one is running.
2. Tracking thread continues processing incoming frames.
3. A later frame can become a keyframe when Local Mapping is ready (BA adjustment interruption increases this chance).

Reason for local BA to be interruptible: map optimization should not indefinitely prevent the system from adding important new observations.

Also, if Local Mapping has been stopped or a stop has been requested for loop correction, keyframe insertion is rejected regardless of these conditions.

# Limitations

- Camera movement must be present, and motion should be significant but smooth:
    - Camera is stationary – no triangulation is possible.
    - Camera moves quickly or unpredictably – features may leave the field of view, features may move far across the image, projection-based matching becomes harder (it searches a neighborhood), constant-velocity prediction can be inaccurate.
    - The camera moves slowly and smoothly – this is generally favorable. Consecutive frames have significant overlap, the motion model works reasonably well, and feature matching is easier. The downside is that very small translation between frames produces weak triangulation geometry.
- Severe blur is destructive for many stages of ORB-SLAM.
    Practical mitigations:
    - shorter exposure time,
    - better lighting
    - stabilization
    - higher frame rate when exposure permits
    - image-quality checks before attempting expensive mapping operations (variance of Laplacian)
    - Sharpening or deblurring can sometimes help but cannot reliably recover information lost to severe blur.
- Large rotation is destructive:
    - Pure rotation ($t=0$) - The essential matrix becomes $E=[\mathbf t]_\times R=0$. Consequently, the essential-matrix constraint cannot recover translation or triangulate new 3D points. If the map already exists, the system may still track the camera orientation using 2D–3D correspondences.
    - Large rotation between consecutive frames - Even with translation, a rapid turn may move many landmarks outside the field of view. Motion prediction can fail, and blur may further reduce the number of correspondences.
- Environment should be sufficiently textured. Solid textures or repeated patterns such as identical windows cause problems.
- Moving objects should be excluded. Most conventional feature-based SLAM pipelines assume that mapped landmarks are stationary.
- Camera view:
    - Forward-facing camera:
        - Advantages: seeing obstacles and landmarks ahead, and retaining landmarks across multiple frames when moving forward.
        - Disadvantages: forward translation. Image motion is small near the focus of expansion, and distant landmarks provide weak triangulation parallax.
    - Downward-facing camera:
        - Advantages: potentially textured ground surface and strong image motion when translating laterally over it.
        - Disadvantages:
            - If the visible scene is almost entirely planar, the image correspondences are well described by a homography: $\mathbf x_2\sim H\mathbf x_1$. Planar geometry can make general 3D reconstruction and motion estimation more ambiguous, especially during initialization. A planar scene is not automatically a failure: planar tracking can work well, and homography-based methods can exploit this structure.
            - If the ground is uniform, such as clean asphalt or a smooth floor, feature detection becomes another problem.
    - Sideways-facing or oblique camera:
        - Advantages: features often produces substantial translational parallax. This can be favorable for triangulation, provided that the scene contains stable landmarks and enough overlap between frames.
        - Disadvantages: depending on conditions, may face some of the problems of forward-facing and downward-facing cameras.


# Other approaches

This file is mainly focused of ORB-SLAM and ORB-SLAM2 implementations. But there are other approaches. They differ mostly by the information and sensors they use.

For example:
- ORB-SLAM3 - Supports monocular, stereo, RGB-D, and visual-inertial configurations, including multi-map management and map merging.
- VINS-Mono / VINS-Fusion - Major reference implementations for tightly coupled visual-inertial estimation. Especially relevant when combining smartphone camera measurements with IMU data.
- DSO (Direct Sparse Odometry) - An important alternative to ORB-style feature matching. It estimates motion by minimizing photometric error over selected image pixels. It is primarily visual odometry, not a complete loop-closing SLAM system.